# Fine-Tuning and Cleaning

This notebook fine-tunes an open-access LLM from [HuggingFace](https://huggingface.co) to correct OCR errors in historical newspaper text. It uses LoRA (Low-Rank Adaptation) with prompt tuning to adjust the model's baseline performance. In other words, fine-tuning is performed using pairs of (prompt + raw OCR text) → corrected transcription using the ground truth CSV from this repository.

For more information about LoRA fine-tuning, see the `/code/README.md`.

**Run cells from top to bottom.**



## For working in Colab

## Connect to the GPU:
In the 'Runtime' menu, click on 'Change runtime type' and select 'T4 GPU' under 'Hardware accelerator'. **NOTE:** you will need to be logged in with your Google account to connect your Drive and to use the GPU. Free access is limited; for larger projects, you may need to consider working on an HPC.

## Connect your Google Drive:
Run the code block below to mount your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install and import dependencies
Fine-tuning requires several additional libraries beyond the base pipeline. Run this cell first.

**Note:** A GPU is required. In Google Colab, enable GPU via **Runtime > Change runtime type > T4 GPU**. For HPC users, you will need to request a GPU node in your SLURM job script — see `HPC/README.md` for more information about how to do this.

**Also note:** Downloading some HuggingFace models requires an account and acceptance of the model's licence terms. Check the HuggingFace model card to see if this is the case for your selection. If so:

1. Create a free account at [huggingface.co](https://huggingface.co)
2. Navigate to the model page (e.g. [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)) and accept the licence
3. Create an access token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) with Read permissions
4. Take note of your token; you will need to paste it below



In [ ]:
!pip install pandas transformers torch peft trl bitsandbytes accelerate datasets tqdm huggingface_hub

In [ ]:
# run if you are using a gated model; otherwise, you can skip this cell
from huggingface_hub import login

hf_token = 'your_huggingface_token_here'  # paste your token here
login(token=hf_token)

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import torch

## 2. Load and split data
This code block imports the dataframe used in the previous steps of the pipeline, then splits it into training and testing datasets.

In [ ]:
df = pd.read_csv('/content/drive/My Drive/THISTLE_project/post_ocr_output.csv')


train_df, test_df = train_test_split(df, test_size=0.8, random_state=13)

## 3. Prepare dataset for fine-tuning

Because each sample in the training dataset is formatted as a prompt with an input-output, we need to restructure it. The cell below creates a function to do so. In this example, it inserts the ProQuest OCR into the prompt as part of the input and the ground truth (manually transcribed article) into the prompt as the output.

You can adjust these inputs and outputs by changing the columns included in the function.

Each training example is formatted as an instruction–response pair:

- **Instruction (input):** the OCR correction prompt with the raw ProQuest OCR text inserted
- **Response (output):** the manually transcribed ground truth

The model learns to map from the (prompt + OCR text) to the corrected transcription.


In [ ]:
from datasets import Dataset

def format_instruction(row):
    return {
        "prompt": f"please correct the OCR without modernising or changing the language and without adding new information {row['ocr']}",
        "completion": row['ground_truth']
    }

train_dataset = Dataset.from_pandas(train_df).map(format_instruction)

## 4. Fine-tuning setup
Set the model, paths and training parameters. The base model is downloaded from HuggingFace. To reduce GPU memory usage, it is loaded in 4-bit quantisation using `bitsandbytes` (QLoRA).

If you do not have `bitsandbytes` installed or prefer to fine-tune without quantisation, set `use_4bit = False` below. Note, however, that doing so will require more GPU memory. LoRA (Low-Rank Adaptation) adds a small number of trainable parameters while keeping all original model weights frozen.

In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct" # insert the model's full label here

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

## 5. Train
This cell runs the fine-tuning. Training time depends on the size of the model and dataset in addition to the GPU type and memory.



In [ ]:
training_args = TrainingArguments(
    output_dir="./llama3-ocr-ft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    save_steps=100,
    logging_steps=10,
    fp16=True
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    dataset_text_field="prompt",
    max_seq_length=1024,
    packing=False
)

trainer.train()

## 6. Deploy on test set
This cell generates corrected text for each row in the validation set using the fine-tuned model. Outputs are also added to the validation dataframe and saved to a CSV for further analysis in later stages.

In [ ]:
def generate_prediction(text):
    prompt = f"please correct the OCR without modernising or changing the language and without adding new information {text}"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the completion part if necessary
    return prediction

tqdm.pandas()
test_df['ft_prediction'] = test_df['original_ocr'].progress_apply(generate_prediction)


## 7. Save results

In [ ]:
test_df.to_csv('/content/drive/My Drive/THISTLE_project/llama_fine_tuned_output.csv')